# Python: Testing Strategies

Course by **Emily Bache** — Technical Coach  

This notebook covers all three modules:
1. Using Pytest to Create Unit Tests
2. Using Test Doubles
3. Integration Tests and Functional Tests

---
## Part 1 — Using Pytest to Create Unit Tests

### What Is an Automated Test?

| | |
|---|---|
| **Your production code** | Has important behaviors and outcomes for users at runtime |
| **An automated test** | Verifies the behavior of (parts of) the code matches expectations |

### Test Automation Strategy — The Pyramid

Automated tests have **different granularity**:  
- The bigger the tests, the fewer of them you have  
- The pyramid axis: how much of the application the test exercises vs. number of tests

```
        /\
       /  \       ← Functional tests  (fewest, exercise most)
      /----\
     /      \     ← Integration tests
    /--------\
   /          \   ← Unit tests        (most, exercise least)
  /____________\
```

> Citation: Seb Rose, "Eviscerating the Test Automation Pyramid"

### Pytest for Test Automation in Python

- The pytest framework makes it easy to write **readable tests**
- It can scale from small unit tests to complex functional testing

> Citation: https://docs.pytest.org/

### What Is a Unit Test?

- A **unit** is a small piece of code with a specific behavior that can be understood separately
- An **automated unit test** verifies the behavior of the unit matches expectations
- Unit tests are the **basis of your strategy** — small, fast tests

### Vocabulary

| Term | Definition |
|---|---|
| **Unit Under Test** | The small piece of code being tested |
| **Test Case** | A single test that exercises one behavior of the unit under test |
| **Test Runner** | Executes test cases and reports results (pass/fail) |

Run pytest from the command line:
```bash
python -m pytest
```

### Exercise: Phonebook

Given a list of names and phone numbers, make a phonebook.  
Determine if it is **consistent**:
- No duplicate numbers
- No number is a prefix of another

**Test list:**
- Look up by name
- Missing name (expect KeyError)
- Consistent phonebook
- Inconsistent phonebook

### Demo: Phonebook — Production Code

In [1]:
class Phonebook:
    def __init__(self):
        self.storage = {}

    def add(self, name, number):
        self.storage[name] = number

    def lookup(self, name):
        return self.storage[name]

    def names(self):
        return self.storage.keys()

    def is_consistent(self):
        for name1, number1 in self.storage.items():
            for name2, number2 in self.storage.items():
                if name1 == name2:
                    continue
                if number1.startswith(number2):
                    return False
        return True

    def find_clashes(self, number):
        clashes = []
        for name, existing_number in self.storage.items():
            if number.startswith(existing_number) or existing_number.startswith(number):
                clashes.append((name, existing_number))
        return clashes

### Assertions in Pytest

- Use the `assert` keyword — pytest rewrites it to give detailed failure messages
- To expect an exception, use `with pytest.raises(...)`

> More info: https://docs.pytest.org/en/stable/how-to/assert.html

### Demo: A First Unit Test with Pytest

In pytest, test files are named `test_*.py` and test functions start with `test_`.

In [2]:
# Simulating pytest with a simple assert — in a real project this goes in tests/test_phonebook.py

pb = Phonebook()
pb.add("Bob", "1234")
assert "1234" == pb.lookup("Bob"), "lookup should return the stored number"
print("test_look_up_by_name PASSED")

test_look_up_by_name PASSED


### Test Fixtures

A **fixture** is shared setup code that creates the objects needed by a test.  
Pytest fixtures are declared with `@pytest.fixture` and injected by parameter name.

```python
import pytest

@pytest.fixture
def phonebook():
    return Phonebook()
```

### Parameterized Tests

Run the same test logic with multiple inputs using `@pytest.mark.parametrize`.

In [3]:
# Demonstrating parameterized logic inline
import pytest

test_cases = [
    (("Bob", "1234"), ("Ted", "1234"), False),   # duplicates
    (("Bob", "1234"), ("Ann", "12345678"), False), # prefix clash
    (("Bob", "1234"), ("Sue", "78955"), True),     # no clash
]

for entry1, entry2, expected in test_cases:
    pb = Phonebook()
    pb.add(*entry1)
    pb.add(*entry2)
    result = pb.is_consistent()
    status = "PASSED" if result == expected else "FAILED"
    print(f"{entry1} + {entry2} → consistent={result}, expected={expected}  [{status}]")

('Bob', '1234') + ('Ted', '1234') → consistent=False, expected=False  [PASSED]
('Bob', '1234') + ('Ann', '12345678') → consistent=False, expected=False  [PASSED]
('Bob', '1234') + ('Sue', '78955') → consistent=True, expected=True  [PASSED]


### Full pytest Test File (for reference)

Save as `tests/test_phonebook.py` alongside a `pyproject.toml` that lists `pytest` as a dependency.

```python
import pytest
from phonebook import Phonebook

@pytest.fixture
def phonebook():
    return Phonebook()

def test_look_up_by_name(phonebook):
    phonebook.add("Bob", "1234")
    assert phonebook.lookup("Bob") == "1234"

def test_lookup_missing_name(phonebook):
    with pytest.raises(KeyError):
        phonebook.lookup("Bob")

def test_is_consistent_empty(phonebook):
    assert phonebook.is_consistent()

def test_is_inconsistent_entry_prefix_of_another(phonebook):
    phonebook.add("Bob", "1234")
    phonebook.add("Ann", "12345678")
    assert not phonebook.is_consistent()

@pytest.mark.parametrize(
    "entry1,entry2,is_consistent",
    [
        [("Bob", "1234"), ("Ted", "1234"), False],
        [("Bob", "1234"), ("Ann", "12345678"), False],
        [("Bob", "1234"), ("Sue", "78955"), True],
    ]
)
def test_is_consistent_two_entries(phonebook, entry1, entry2, is_consistent):
    phonebook.add(*entry1)
    phonebook.add(*entry2)
    assert phonebook.is_consistent() == is_consistent
```

### Unit Test Strategy — Design Principles

**Test-Driven — Design from Usage:**
1. Pick the **easiest behavior** on your test list
2. Write a test that **fails** because that behavior is not implemented yet
3. Focus on the **behavior**, not the implementation

Good unit tests are:
- **Fast** — run in milliseconds
- **Independent** — no shared state between tests
- **Readable** — named after the behavior they verify
- **Deterministic** — same result every run

---
## Part 2 — Using Test Doubles

### When to Use a Test Double

Use a test double when:
- The code you want to test **has a dependency**
- For some reason you **don't want to use the real one**

Replace the dependency with a test double, like using a **stunt double** instead of the real actor.

### Test Doubles Enable Smaller, Faster Tests

The basis of your strategy is small, fast tests.  
**Replace slow or awkward code** (I/O, network, databases) with a test double.

**Slow and awkward** dependencies:
- Remote services: API calls, queues, microservices
- Databases, caches, filesystem
- Hardware: system clock, random number generator

### Kinds of Test Double

| Kind | Description |
|---|---|
| **Dummy** | Passed as argument but never actually used |
| **Stub** | Returns a configured answer; no real logic |
| **Fake** | A working simplified implementation (e.g. in-memory DB) |
| **Spy** | Records calls so you can verify them later |
| **Mock** | Pre-programmed with expected calls and verification |

> Citation: Gerard Meszaros, "XUnit Test Patterns", http://xunitpatterns.com/Test%20Double.html

### Common Things to Replace with a Fake

| Real dependency | Replace with |
|---|---|
| File | `StringIO` |
| Database | In-memory database (e.g. SQLite `:memory:`) |
| Web Server | Lightweight test server (e.g. Flask test client) |

### Exercise: Enterprise Phonebook

The Enterprise Phonebook extends the basic Phonebook with:
- **Security Authorisation** — only authorized users can look up entries
- **Batch Update** — load entries from a CSV file
- **Inconsistent Entry Reporting** — alert when an inconsistent entry is attempted
- **Event Logging** — log every lookup

In [4]:
import csv
import dataclasses
from typing import Tuple


class EnterprisePhonebook:
    def __init__(self, phonebook, authorizer, alerter, enterprise_logger):
        self.phonebook = phonebook
        self.authorizer = authorizer
        self.alerter = alerter
        self.logger = enterprise_logger

    def lookup(self, name):
        authorized = self.authorizer.is_authorized()
        if not authorized:
            raise RuntimeError("unauthorized lookup")
        self.logger.info(name)
        return self.phonebook.lookup(name)

    def add_from_file(self, filename):
        with open(filename) as file:
            self.add_data(file)

    def add_data(self, file):
        name = "Name"
        number = "Phone Number"
        expected_header = "%s,%s" % (name, number)
        actual_header = file.readline().strip()
        assert actual_header == expected_header
        reader = csv.DictReader(file, [name, number])
        for row in reader:
            self.phonebook.add(row[name], row[number])

    def add(self, name, number):
        clashes = self.phonebook.find_clashes(number)
        if not clashes:
            self.phonebook.add(name, number)
        else:
            for entry in clashes:
                event = BadPhonebookEntryEvent(len(self.phonebook.storage), name, number, entry)
                self.alerter.send_alert(event)


@dataclasses.dataclass
class BadPhonebookEntryEvent:
    size: int
    new_name: str
    new_number: str
    problem: Tuple[str, str]

### Demo: Using a Stub

A **stub** provides a canned response for `is_authorized()` so we can test the `EnterprisePhonebook.lookup` without a real auth service.

In [5]:
from unittest.mock import create_autospec, Mock


class SecurityClearanceAuthorizer:
    def is_authorized(self) -> bool:
        raise NotImplementedError


# Create a stub that always returns True
stub_authorizer = create_autospec(SecurityClearanceAuthorizer)
stub_authorizer.is_authorized = Mock(return_value=True)

phonebook = Phonebook()
phonebook.add("Bob", "1234")

enterprise = EnterprisePhonebook(phonebook, stub_authorizer, Mock(), Mock())
result = enterprise.lookup("Bob")
print(f"lookup result: {result}")
assert result == "1234"
print("test_lookup_authorized PASSED")

lookup result: 1234
test_lookup_authorized PASSED


### Demo: Using a Fake (StringIO)

A **fake** is a real but simplified implementation.  
`StringIO` acts as a file without touching the filesystem.

In [6]:
from io import StringIO
from textwrap import dedent

fake_file = StringIO(dedent("""\
Name,Phone Number
Bob,1234
Ann,6789"""))

phonebook = Phonebook()
enterprise = EnterprisePhonebook(phonebook, stub_authorizer, Mock(), Mock())
enterprise.add_data(fake_file)

assert phonebook.storage == {"Bob": "1234", "Ann": "6789"}
print("test_update_from_file PASSED")

test_update_from_file PASSED


### Demo: Using a Spy

A **spy** records the calls made to it so you can assert on side effects.

In [7]:
import unittest


class InconsistentPhonebookAlerter:
    def send_alert(self, event):
        raise NotImplementedError


spy_alerter = create_autospec(InconsistentPhonebookAlerter)

phonebook = Phonebook()
enterprise = EnterprisePhonebook(phonebook, stub_authorizer, spy_alerter, Mock())
enterprise.add("Bob", "12345")
enterprise.add("Sid", "12346")
enterprise.add("Ted", "1234")  # clashes with Bob and Sid

expected_calls = [
    unittest.mock.call(BadPhonebookEntryEvent(2, "Ted", "1234", ("Bob", "12345"))),
    unittest.mock.call(BadPhonebookEntryEvent(2, "Ted", "1234", ("Sid", "12346"))),
]
spy_alerter.send_alert.assert_has_calls(expected_calls)
print("test_alert_inconsistent_entry_attempts PASSED")

test_alert_inconsistent_entry_attempts PASSED


### Demo: Using a Dummy

A **dummy** is an object passed to satisfy a parameter signature but never actually used.

In [8]:
# When testing add_data, we don't need the alerter or authorizer to do anything real
dummy_authorizer = Mock()
dummy_alerter = Mock()
dummy_logger = Mock()

phonebook = Phonebook()
enterprise = EnterprisePhonebook(phonebook, dummy_authorizer, dummy_alerter, dummy_logger)

# The dummies are never called; add_data only uses phonebook internally
fake_file = StringIO(dedent("""\
Name,Phone Number
Alice,9999"""))

enterprise.add_data(fake_file)
assert "Alice" in phonebook.storage
print("test_add_data_with_dummies PASSED")

test_add_data_with_dummies PASSED


---
## Part 3 — Integration Tests and Functional Tests

### The Test Pyramid — All Three Layers

```
        /\
       /  \          ← Functional tests  (fewest)
      /----\
     /      \        ← Narrow integration tests
    /--------\
   /          \      ← Unit tests        (most)
  /____________\
```

**Python testing strategy:**
- Unit tests: verify domain logic in isolation
- Narrow integration tests: verify adapters can talk to external services
- Functional tests: verify a complete user workflow end-to-end

### Integration Testing

**Integration tests check pieces work together.**

They include out-of-process calls and I/O:
- Remote services: API calls, queues, microservices
- Databases, caches, filesystem  
- Hardware: system clock, random number generator

### Domain-Centric Architecture (Hexagonal / Ports & Adapters)

```
                    Outside world
                         │
          Inbound ───────┼──────── Outbound
        (Requests)  ┌────┴────┐   (Events)
                    │  App    │
                    │ Domain  │
                    │  Code   │
                    └─────────┘
                 Ports & Adapters Layer
```

- **Inbound ports** — accept requests from outside (e.g. HTTP handlers)
- **Outbound ports** — send events/calls to other services (e.g. HTTP clients, DB adapters)
- **Adapter classes** — implement the port interface for a specific technology

> Citation: Alistair Cockburn, "The Hexagonal (Ports and Adapters) Architecture"  
> Citation: Ted M. Young, "More Testable Code with Hexagonal Architecture"

### Narrow Integration Tests

**Narrow integration tests** are:
- Smaller and faster than broad integration tests  
- Check that **an adapter can talk to an external service**

There are two kinds:
- **Outbound port test** — checks the adapter that sends events/calls outward  
- **Inbound port test** — checks the adapter that receives requests from outside

Together they create **overlapping small, fast tests** that cover the boundaries without running the full stack.

### Demo: Narrow Integration Tests for Outbound Ports

Testing that the `alerts` adapter correctly calls the external `AuthAndAlertService`.

```python
# tests/integration/test_alerts.py  (phonebook_demo_m3)
import pytest
from alerts import InconsistentPhonebookAlerter

# Requires the AuthAndAlertService to be running locally
@pytest.mark.integration
def test_send_alert_reaches_service(running_alert_service):
    alerter = InconsistentPhonebookAlerter(base_url=running_alert_service)
    event = ...
    alerter.send_alert(event)
    # verify the service received the alert
```

### Demo: Narrow Integration Tests for Inbound Ports

Testing that the `authorization` adapter correctly receives and interprets requests.

```python
# tests/integration/test_authorizer.py  (phonebook_demo_m3)
import pytest
from authorization import SecurityClearanceAuthorizer

@pytest.mark.integration
def test_authorized_user(running_auth_service):
    authorizer = SecurityClearanceAuthorizer(base_url=running_auth_service)
    assert authorizer.is_authorized()
```

### Functional Tests

**Functional tests set up everything needed for a whole user workflow.**

They exercise the full application from end to end — including all adapters, the domain code, and any external services.

The implementation depends on your architecture:
- Tablet/mobile app → UI automation
- Desktop application → GUI testing framework
- Web page → Selenium / Playwright

### Demo: Functional Tests for Enterprise Phonebook

```python
# tests/functional/test_enterprise_phonebook.py  (phonebook_demo_m3)
import pytest

@pytest.mark.functional
def test_authorized_user_can_look_up_entry(running_system):
    # Arrange: start all services, populate phonebook
    client = running_system.phonebook_client()
    client.add("Bob", "1234")

    # Act
    result = client.lookup("Bob")

    # Assert
    assert result == "1234"
```

### Overlapping Tests — the Full Strategy

The strategy creates overlapping coverage:

```
┌─────────────────────────────────────────────────────┐
│  Functional Test (orange boundary)                  │
│  ┌──────────────┐   ┌──────────────────────────┐   │
│  │  Unit Tests  │   │ Narrow Integration Tests  │   │
│  │  (green)     │   │ (green, between adapters) │   │
│  └──────────────┘   └──────────────────────────┘   │
└─────────────────────────────────────────────────────┘
```

Each test type has a specific job:
- **Unit tests** — verify domain logic cheaply and quickly
- **Narrow integration tests** — verify each adapter boundary works with its real service
- **Functional tests** — verify entire user workflows work end to end

---
## Summary

| Topic | Key Takeaway |
|---|---|
| pytest | Easy to write readable tests; scales from unit to functional |
| Assertions | Use plain `assert`; use `pytest.raises` for expected exceptions |
| Fixtures | `@pytest.fixture` for shared setup; injected by parameter name |
| Parametrize | `@pytest.mark.parametrize` to run one test with many inputs |
| Test doubles | Stub (returns value), Fake (simplified impl), Spy (records calls), Dummy (unused) |
| Hexagonal arch | Separate domain code from adapters using inbound/outbound ports |
| Narrow integration tests | Test only the adapter + its real service, not the whole system |
| Functional tests | Full user workflow; use sparingly, they are slow and costly |

**Python testing pyramid:**
```
Functional tests     ← fewest
Narrow integration
Unit tests           ← most
```

Demo code lives in:
- `01-using-pytest-to-create-unit-tests/demos/phonebook_demo/`
- `02-using-test-doubles/demos/phonebook_demo_m2/`
- `03-integration-tests-and-functional-tests/demos/phonebook_demo_m3/`